# MetaTT の再現：表 2 の最小実験（RoBERTa-base × MetaTT-4D r=8 × MRPC・RTE）

対象論文：Lopez-Piqueres et al., "MetaTT: A Global Tensor-Train Adapter for Parameter-Efficient Fine-Tuning", TMLR 2026（arXiv:2506.09105）。
論文の中でいちばん小さい実験（表 2 の 1 マス）で、MetaTT-4D の正解率が論文どおりになるかを確かめます。比較手法（LoRA など）は回しません。

**使い方**：「ランタイム → ランタイムのタイプを変更」で **A100 GPU・ハイメモリ** を選び、「ランタイム → すべてのセルを実行」を押すだけです。コードはこのノートブックの中にあるので、アップロードは不要です。結果は Google ドライブの `MyDrive/MetaTT_repro/results` に保存され、途中で切れても、もう一度「すべてのセルを実行」を押せば終わったランは飛ばします（途中のランは最初からやり直し）。

**設定（論文の値）**

| 項目 | 値 | 論文の出どころ（v2 / v3） |
|---|---|---|
| モデル | RoBERTa-base（`FacebookAI/roberta-base`） | 表 2 |
| アダプタ | MetaTT-4D、ランク 8、Q と V に適用、学習するパラメータ 13,184（論文 13k） | 表 2、付録 D.2 / G.2 |
| 初期化 | G1 はゼロ、G2・G3・G4 は単位行列 | 付録 D.4 / G.4 |
| 学習 | 分類ヘッドは凍結、20 エポック、系列長 256、weight decay 0 | 3.1 節、付録 D.4 / G.4 |
| シード | 33305628、2025、42 | 付録 D.1 / G.1 |
| MRPC | α = 0.5、学習率 1e-3、バッチ 8 | 表 6 / 表 14 |
| RTE | α = 4、学習率 1e-3、バッチ 16 | 表 6 / 表 14 |
| 目標値 | MRPC 87.6 ± 0.2、RTE 72.9 ± 0.5（検証データの正解率、± は標準誤差） | 表 2 |

**論文に書かれていない・食い違っている設定（こちらで決めたこと）**
- warmup ratio：論文の GLUE の段落は 0.6、ほかの実験はすべて 0.06。**両方**回します（0.06 → 0.6 の順）。
- 表 6 / 表 14 の該当行はランク「4」と印刷されていますが、13k のパラメータ数はランク 8 でしか出ないため、ランク 8 の行として使います。
- 最適化は HF Trainer の既定値（AdamW、線形スケジュール、勾配クリップ 1.0）。報告値は「20 エポック中の最良」と「最終エポック」の両方を出します。
- 数値の精度は fp32（TF32 も使わない）。論文には書かれていません。

**ラン数**：2 タスク × 3 シード × warmup 2 通り = 12 本（1 本は MRPC 9,180 ステップ、RTE 3,120 ステップ）。同時に数本ずつ流します。時間はまだ測っていないので、最初のランが終わったときの「[x.x min]」で確かめてください。

## 設定とセットアップ
必要なら `USE_DRIVE`・`TASKS`・`WARMUPS`・`HOURS` だけ変えてください。

In [ ]:
import os, sys, time, subprocess, torch
from google.colab import drive

USE_DRIVE = True                                   # False: 結果を /content に置く（切断すると消える）
OUT       = '/content/drive/MyDrive/MetaTT_repro/results' if USE_DRIVE else '/content/results'
TASKS     = 'mrpc,rte'                             # 'mrpc' だけ、'rte' だけにもできる
WARMUPS   = '0.06,0.6'                             # '0.06' だけにすると 6 本で済む
PRECISION = 'fp32'                                 # 'tf32' / 'bf16' にすると速いが、論文の条件からは外れる（別フォルダに保存）
HOURS     = 10                                     # この時間を過ぎたら新しいランを始めない
CODE      = '/content/metatt_code'

if USE_DRIVE:
    drive.mount('/content/drive')
os.makedirs(OUT, exist_ok=True)
for d in ('metatt', 'metatt_repro', 'tests'):
    os.makedirs(f'{CODE}/{d}', exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
assert torch.cuda.is_available(), 'GPU がありません。「ランタイム → ランタイムのタイプを変更」で A100 GPU を選んでください'
gpu = torch.cuda.get_device_name(0)
gpu_gb = torch.cuda.get_device_properties(0).total_memory / 2**30
cpus = os.cpu_count() or 2
PARALLEL = max(1, min(int((gpu_gb - 4) // 4), max(1, cpus // 2), 6))   # 1 つの GPU で同時に流すラン数
print(f'GPU: {gpu}（{gpu_gb:.0f} GB）, CPU {cpus} 個 -> 同時実行 {PARALLEL} 本 | 精度 {PRECISION}')
print('結果の保存先:', OUT, '| コードの版:', '85a22862cd')
subprocess.run('pip -q install "datasets>=2.18"', shell=True, check=False)

## コード（変更不要）
論文の擬似コード（付録 E / H）と本文の式から書いた実装です。リポジトリ paper-reproductions の `papers/2026-tmlr-metatt/` にあるものと同じで、`metatt/` が論文の手法（MetaTT-4D）、`metatt_repro/` が再現実験、`tests/` が CPU のテストです。下のセルが `/content/metatt_code/` に書き出します。

In [ ]:
%%writefile /content/metatt_code/metatt/__init__.py
"""MetaTT (Lopez-Piqueres et al., TMLR 2026, arXiv:2506.09105): a global tensor-train adapter."""
from .adapter import (MetaTT4D, MetaTT4DConfig, MetaTTLinear, apply_metatt, count_trainable, find_targets,
                      metatt4d_num_params)

__all__ = ["MetaTT4D", "MetaTT4DConfig", "MetaTTLinear", "apply_metatt", "count_trainable", "find_targets",
           "metatt4d_num_params"]


In [ ]:
%%writefile /content/metatt_code/metatt/adapter.py
"""MetaTT-4D: one tensor train (TT) shared by every adapted linear map of a transformer.

Lopez-Piqueres et al., "MetaTT: A Global Tensor-Train Adapter for Parameter-Efficient Fine-Tuning",
TMLR 2026 (arXiv:2506.09105). Written from the paper: Sec. 2.3, Eq. (5) and the pseudo-code of
Appendix E (v2) / Appendix H (v3), Algorithms 2-5.

For layer l and projection type m (e.g. m = 0 for Q, 1 for V) the adapted map is (Eq. (5))

    y = x W_{l,m}^T + b + alpha * x G1 G2[l] G3[m] G4

    G1 in R^{D_in x r1}      input core (shared by every layer and type)
    G2 in R^{L x r1 x r2}    layer core (slice G2[l] is an r1 x r2 matrix)
    G3 in R^{M x r2 x r3}    projection-type core
    G4 in R^{r3 x D_out}     output core

Trainable parameters: D_in r1 + L r1 r2 + M r2 r3 + r3 D_out, i.e. 2 D r + (L + M) r^2 for equal ranks.

Differences from the paper's pseudo-code (none change the computation):
  * G2 and G3 are stored as one 3-way tensor each instead of a ParameterList of r x r matrices.
    Element-wise optimisers (AdamW) and global-norm clipping treat both layouts identically.
  * The pseudo-code's typos are fixed (`self.M_2` vs `self.M2`; `ParameterList[...]` for `ParameterList([...])`).
"""
from __future__ import annotations

import re
from dataclasses import dataclass

import torch
from torch import nn

INIT_CODES = ("ze", "id", "no")       # zeros / identity on every matrix slice / normal(0, init_std)


@dataclass
class MetaTT4DConfig:
    """Algorithm 2. `rank` is one int (all three TT bonds equal, as in the paper) or (r1, r2, r3)."""
    rank: int | tuple[int, int, int] = 8
    alpha: float = 1.0                                  # scaling of the adapter output (Eq. (5)); not divided by r
    target_modules: tuple[str, ...] = ("query", "value")  # leaf names of the nn.Linear modules to adapt
    init: str = "ze-id-id-id"                            # init of G1-G2-G3-G4 (Appendix D.4 / G.4, Fig. 5)
    init_std: float = 0.2                                # std of the "no" init (Appendix D.4)

    @property
    def ranks(self) -> tuple[int, int, int]:
        r = (self.rank,) * 3 if isinstance(self.rank, int) else tuple(self.rank)
        if len(r) != 3 or min(r) < 1:
            raise ValueError(f"rank must be a positive int or three positive ints, got {self.rank!r}")
        return r

    @property
    def init_codes(self) -> tuple[str, str, str, str]:
        codes = tuple(self.init.split("-"))
        if len(codes) != 4 or any(c not in INIT_CODES for c in codes):
            raise ValueError(f"init must be four of {INIT_CODES} joined by '-', got {self.init!r}")
        return codes


def metatt4d_num_params(d_in: int, d_out: int, num_layers: int, num_types: int, rank) -> int:
    """Closed-form parameter count (Sec. 2.3, 'Complexity Analysis')."""
    r1, r2, r3 = MetaTT4DConfig(rank=rank).ranks
    return d_in * r1 + num_layers * r1 * r2 + num_types * r2 * r3 + r3 * d_out


class MetaTT4D(nn.Module):
    """The four TT cores (Algorithm 3) and the adapter term alpha * x G1 G2[l] G3[m] G4."""

    def __init__(self, d_in: int, d_out: int, num_layers: int, num_types: int, cfg: MetaTT4DConfig):
        super().__init__()
        r1, r2, r3 = cfg.ranks
        self.cfg = cfg
        self.alpha = float(cfg.alpha)
        self.G1 = nn.Parameter(torch.empty(d_in, r1))
        self.G2 = nn.Parameter(torch.empty(num_layers, r1, r2))
        self.G3 = nn.Parameter(torch.empty(num_types, r2, r3))
        self.G4 = nn.Parameter(torch.empty(r3, d_out))
        self.reset_parameters()

    def cores(self) -> tuple[nn.Parameter, ...]:
        return self.G1, self.G2, self.G3, self.G4

    @torch.no_grad()
    def reset_parameters(self) -> None:
        for core, code in zip(self.cores(), self.cfg.init_codes):
            if code == "ze":
                core.zero_()
            elif code == "no":
                core.normal_(0.0, self.cfg.init_std)
            else:                                        # "id": identity on every matrix slice (nn.init.eye_)
                core.zero_()
                mats = core if core.dim() == 3 else core.unsqueeze(0)
                k = min(mats.shape[-2], mats.shape[-1])
                idx = torch.arange(k, device=core.device)
                mats[:, idx, idx] = 1.0

    def forward(self, x: torch.Tensor, layer: int, mtype: int) -> torch.Tensor:
        # Algorithm 4: (((X M1) M2) M3) M4, cheapest order when the rank is below the number of tokens
        h = x @ self.G1
        h = h @ self.G2[layer]
        h = h @ self.G3[mtype]
        return self.alpha * (h @ self.G4)

    def delta_weight(self, layer: int, mtype: int) -> torch.Tensor:
        """Dense update Delta W_{l,m} (shape D_in x D_out) so that the adapter term equals x @ Delta W."""
        return self.alpha * (self.G1 @ self.G2[layer] @ self.G3[mtype] @ self.G4)

    def extra_repr(self) -> str:
        (d_in, r1), (num_layers, _, r2), (num_types, _, r3), (_, d_out) = (c.shape for c in self.cores())
        return (f"D_in={d_in}, L={num_layers}, M={num_types}, D_out={d_out}, ranks=({r1}, {r2}, {r3}), "
                f"alpha={self.alpha}, init={self.cfg.init}")


class MetaTTLinear(nn.Module):
    """Algorithm 4: frozen nn.Linear plus the shared MetaTT term for its (layer, type) slice."""

    def __init__(self, base: nn.Linear, adapter: MetaTT4D, layer: int, mtype: int):
        super().__init__()
        self.base = base
        # Shared by every wrapped layer and registered once on the model root (as `metatt`);
        # stored outside nn.Module's registry so the cores are not duplicated in state_dict().
        object.__setattr__(self, "_adapter", adapter)
        self.layer = layer
        self.mtype = mtype

    @property
    def adapter(self) -> MetaTT4D:
        return self._adapter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.base(x) + self._adapter(x, self.layer, self.mtype)

    def extra_repr(self) -> str:
        return f"layer={self.layer}, type={self.mtype}"


_INDEX = re.compile(r"\.(\d+)\.")


def find_targets(model: nn.Module, target_modules) -> list[tuple[str, int, int, nn.Linear]]:
    """(module name, layer number, type index, module) for every nn.Linear named like `...layer.<l>....<target>`.

    The layer number is the last integer component of the module path, which is the transformer block index
    for RoBERTa/BERT (`roberta.encoder.layer.3.attention.self.query`) and Llama (`model.layers.3.self_attn.q_proj`).
    """
    out = []
    for name, mod in model.named_modules():
        leaf = name.rsplit(".", 1)[-1]
        if isinstance(mod, nn.Linear) and leaf in target_modules:
            nums = _INDEX.findall(name + ".")
            if nums:
                out.append((name, int(nums[-1]), list(target_modules).index(leaf), mod))
    return out


def apply_metatt(model: nn.Module, cfg: MetaTT4DConfig) -> MetaTT4D:
    """Algorithm 5: freeze the whole model (classifier head included), build one MetaTT-4D shared by every
    target linear map and swap each of those maps for a MetaTTLinear. Returns the adapter, which is also
    registered on the model as `model.metatt`, so its four cores are the only trainable parameters."""
    if hasattr(model, "metatt"):
        raise ValueError("model already has a MetaTT adapter")
    targets = find_targets(model, cfg.target_modules)
    if not targets:
        raise ValueError(f"no nn.Linear named {cfg.target_modules} inside numbered blocks")
    layers = sorted({t[1] for t in targets})
    types = sorted({t[2] for t in targets})
    if types != list(range(len(cfg.target_modules))):
        missing = [m for i, m in enumerate(cfg.target_modules) if i not in types]
        raise ValueError(f"target modules not found: {missing}")
    seen = {(t[1], t[2]) for t in targets}
    if len(seen) != len(targets) or len(seen) != len(layers) * len(types):
        raise ValueError("every block must contain each target module exactly once")
    d_in = {t[3].in_features for t in targets}
    d_out = {t[3].out_features for t in targets}
    if len(d_in) != 1 or len(d_out) != 1:
        raise ValueError(f"shared G1/G4 need equal shapes, got in={d_in} out={d_out}")

    for p in model.parameters():
        p.requires_grad_(False)
    ref = targets[0][3].weight
    adapter = MetaTT4D(d_in.pop(), d_out.pop(), len(layers), len(types), cfg).to(ref.device)
    model.add_module("metatt", adapter)
    pos = {l: i for i, l in enumerate(layers)}
    for name, layer, mtype, mod in targets:
        parent_name, leaf = name.rsplit(".", 1)
        setattr(model.get_submodule(parent_name), leaf, MetaTTLinear(mod, adapter, pos[layer], mtype))
    return adapter


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
%%writefile /content/metatt_code/metatt_repro/__init__.py
"""Reproduction of MetaTT (arXiv:2506.09105) Table 2, RoBERTa-base, MetaTT-4D rank 8 on GLUE MRPC and RTE."""
import os
import sys

try:                                   # the method lives in ../../method/metatt in the repository layout
    import metatt  # noqa: F401
except ImportError:
    sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(os.path.dirname(os.path.abspath(__file__)))),
                                    "method"))


In [ ]:
%%writefile /content/metatt_code/metatt_repro/paper.py
"""Settings and reported numbers taken from the paper (arXiv:2506.09105; v2 = 2025-11, v3 = TMLR version 2026-07).

Section and table numbers are given for v2 / v3. Values below are identical in both versions.
"""

# Table 2, RoBERTa-base, MetaTT-4D, rank 8 (13k trainable parameters). Accuracy on the GLUE validation set.
# The number in parentheses in the paper is the standard error (rounded up to one significant digit).
PAPER_RESULTS = {
    "mrpc": {"value": 87.6, "se": 0.2, "metric": "accuracy"},
    "rte": {"value": 72.9, "se": 0.5, "metric": "accuracy"},
}
PAPER_PARAMS_K = 13                                   # Table 2 "Param x 10^3"

# Not from the paper: accuracy of always predicting the majority class of the GLUE validation set
# (MRPC 279/408 "equivalent", RTE 146/277 "not_entailment"). A run stuck at this value has learned nothing.
MAJORITY_BASELINE = {"mrpc": 68.38, "rte": 52.71}

# Table 6 (v2) / Table 14 (v3), first RoBERTa-base block. The block is printed with rank "4", but Table 2 has no
# rank-4 row and its 13k parameters require rank 8 (rank 4 gives 6,368), so we read the block as the rank-8 one.
TASK_HPARAMS = {
    "mrpc": {"alpha": 0.5, "lr": 1e-3, "batch_size": 8},
    "rte": {"alpha": 4.0, "lr": 1e-3, "batch_size": 16},
}

SEEDS = {
    "FacebookAI/roberta-base": (33305628, 2025, 42),  # Appendix D.1 / G.1
    "FacebookAI/roberta-large": (56346, 2025, 42),
}

COMMON = {
    "model": "FacebookAI/roberta-base",
    "rank": 8,
    "target_modules": ("query", "value"),   # Appendix D.2 / G.2: every main-text result adapts Q and V
    "init": "ze-id-id-id",                  # Appendix D.4 / G.4: G1 = 0, other cores identity ("majority of experiments")
    "epochs": 20,                           # Appendix D.4 / G.4: final evaluations run 20 epochs, 3 trials
    "max_len": 256,                         # Appendix D.4 / G.4 (grid paragraph)
    "weight_decay": 0.0,                    # same paragraph
    "max_grad_norm": 1.0,                   # not stated for GLUE; HF Trainer default (the paper uses the Trainer, D.3 / G.3)
}

# Warm-up ratio: the GLUE paragraph prints 0.6 (v2 and v3); every other experiment of the paper uses 0.06.
# We run both: 0.06 (likely intended) and 0.6 (as printed).
WARMUPS = (0.06, 0.6)

# Table 2 MetaTT-4D rows, used by count.py: (model, rank) -> parameters x 10^3 as printed.
TABLE2_PARAMS_K = {
    ("FacebookAI/roberta-base", 8): 13, ("FacebookAI/roberta-base", 24): 45, ("FacebookAI/roberta-base", 64): 156,
    ("FacebookAI/roberta-large", 16): 39, ("FacebookAI/roberta-large", 32): 92,
}


In [ ]:
%%writefile /content/metatt_code/metatt_repro/glue.py
"""GLUE data (Hugging Face `nyu-mll/glue`) and metrics, without scipy/sklearn.

The GLUE test labels are hidden (-1), so, as in LoRA / LoTR / the paper, models are evaluated on the validation split.
"""
from __future__ import annotations

import numpy as np
import torch

# task -> (text keys, number of labels, validation split, main metric)
TASKS = {
    "mrpc": (("sentence1", "sentence2"), 2, "validation", "accuracy"),
    "rte": (("sentence1", "sentence2"), 2, "validation", "accuracy"),
    "cola": (("sentence", None), 2, "validation", "mcc"),
    "sst2": (("sentence", None), 2, "validation", "accuracy"),
    "qnli": (("question", "sentence"), 2, "validation", "accuracy"),
    "qqp": (("question1", "question2"), 2, "validation", "accuracy"),
    "mnli": (("premise", "hypothesis"), 3, "validation_matched", "accuracy"),
    "stsb": (("sentence1", "sentence2"), 1, "validation", "spearman"),
}


def num_labels(task: str) -> int:
    return TASKS[task][1]


def main_metric(task: str) -> str:
    return TASKS[task][3]


def load(task: str, tokenizer, max_len: int, limit_train: int | None = None, limit_eval: int | None = None):
    """Tokenised train and validation sets as dicts of python lists (input_ids, labels)."""
    from datasets import load_dataset

    (k1, k2), _, val_split, _ = TASKS[task]
    ds = load_dataset("nyu-mll/glue", task)
    out = {}
    for split, limit in (("train", limit_train), (val_split, limit_eval)):
        d = ds[split]
        if limit:
            d = d.select(range(min(limit, len(d))))
        a = list(d[k1])
        b = list(d[k2]) if k2 else None
        enc = tokenizer(a, b, truncation=True, max_length=max_len)
        out["train" if split == "train" else "eval"] = {"input_ids": enc["input_ids"], "labels": list(d["label"])}
    return out


def collate(examples: dict, idx, pad_id: int, regression: bool = False) -> dict:
    """Pad the selected examples to the longest one (dynamic padding, as HF's DataCollatorWithPadding)."""
    seqs = [examples["input_ids"][i] for i in idx]
    n = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), n), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), n), dtype=torch.long)
    for j, s in enumerate(seqs):
        ids[j, :len(s)] = torch.tensor(s, dtype=torch.long)
        mask[j, :len(s)] = 1
    lab = [examples["labels"][i] for i in idx]
    labels = torch.tensor(lab, dtype=torch.float if regression else torch.long)
    return {"input_ids": ids, "attention_mask": mask, "labels": labels}


def _rank(x: np.ndarray) -> np.ndarray:
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x), dtype=float)
    ranks[order] = np.arange(len(x), dtype=float)
    for v in np.unique(x):                         # average ranks of ties
        m = x == v
        if m.sum() > 1:
            ranks[m] = ranks[m].mean()
    return ranks


def metrics(task: str, preds: np.ndarray, labels: np.ndarray) -> dict:
    """Percentages, as in the paper's tables. preds are class ids (or scores for STS-B)."""
    if task == "stsb":
        p = float(np.corrcoef(preds, labels)[0, 1])
        s = float(np.corrcoef(_rank(preds), _rank(labels))[0, 1])
        return {"pearson": 100 * p, "spearman": 100 * s}
    out = {"accuracy": 100 * float((preds == labels).mean())}
    if task in ("mrpc", "qqp"):
        tp = float(((preds == 1) & (labels == 1)).sum())
        fp = float(((preds == 1) & (labels == 0)).sum())
        fn = float(((preds == 0) & (labels == 1)).sum())
        out["f1"] = 100 * (2 * tp / (2 * tp + fp + fn) if tp else 0.0)
    if task == "cola":
        tp = float(((preds == 1) & (labels == 1)).sum()); tn = float(((preds == 0) & (labels == 0)).sum())
        fp = float(((preds == 1) & (labels == 0)).sum()); fn = float(((preds == 0) & (labels == 1)).sum())
        den = np.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
        out["mcc"] = 100 * ((tp * tn - fp * fn) / den if den else 0.0)
    return out


In [ ]:
%%writefile /content/metatt_code/metatt_repro/train.py
"""One fine-tuning run: RoBERTa + MetaTT-4D on one GLUE task with one seed.

The paper trains with the Hugging Face Trainer (Appendix D.3 / G.3). This loop reproduces the Trainer's defaults
explicitly so that the run does not depend on the installed transformers version:
  AdamW (betas 0.9/0.999, eps 1e-8), linear decay with linear warm-up over ceil(warmup_ratio * total) steps,
  gradient clipping at global norm 1.0, data reshuffled every epoch, no dropped last batch, dropout on during
  training, evaluation on the validation set after every epoch.

Usage: python -m metatt_repro.train --cfg '<json of RunConfig>' --out <run dir>
"""
from __future__ import annotations

import argparse
import contextlib
import json
import math
import os
import platform
import random
import sys
import time
import traceback
from dataclasses import asdict, dataclass

import numpy as np
import torch

from . import glue
from metatt import MetaTT4DConfig, apply_metatt, count_trainable, metatt4d_num_params


@dataclass
class RunConfig:
    task: str
    seed: int
    alpha: float
    lr: float
    batch_size: int
    warmup_ratio: float
    model: str = "FacebookAI/roberta-base"
    rank: int = 8
    target_modules: tuple = ("query", "value")
    init: str = "ze-id-id-id"
    epochs: int = 20
    max_len: int = 256
    weight_decay: float = 0.0
    max_grad_norm: float = 1.0
    precision: str = "fp32"            # fp32 (no TF32), tf32 (TF32 matmuls), bf16 (bf16 autocast)
    eval_batch_size: int = 64
    limit_train: int | None = None     # smoke tests only
    limit_eval: int | None = None

    @classmethod
    def from_dict(cls, d: dict) -> "RunConfig":
        d = dict(d)
        d["target_modules"] = tuple(d.get("target_modules", cls.target_modules))
        return cls(**d)


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed % 2**32)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def set_precision(precision: str) -> None:
    if precision not in ("fp32", "tf32", "bf16"):
        raise ValueError(f"unknown precision {precision!r}")
    tf32 = precision != "fp32"
    torch.backends.cuda.matmul.allow_tf32 = tf32
    torch.backends.cudnn.allow_tf32 = tf32


def autocast(precision: str, device: torch.device):
    if precision == "bf16" and device.type == "cuda":
        return torch.autocast("cuda", dtype=torch.bfloat16)
    return contextlib.nullcontext()


def linear_schedule(warmup_steps: int, total_steps: int):
    """Same formula as transformers.get_linear_schedule_with_warmup."""
    def f(step: int) -> float:
        if step < warmup_steps:
            return step / max(1, warmup_steps)
        return max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))
    return f


@torch.no_grad()
def frozen_checksum(model: torch.nn.Module) -> float:
    return float(sum(p.detach().double().sum().item() for p in model.parameters() if not p.requires_grad))


@torch.no_grad()
def evaluate(model, data: dict, task: str, device, pad_id: int, batch_size: int, precision: str) -> dict:
    model.eval()
    reg = glue.num_labels(task) == 1
    n = len(data["labels"])
    preds = []
    for s in range(0, n, batch_size):
        batch = glue.collate(data, range(s, min(n, s + batch_size)), pad_id, reg)
        batch.pop("labels")
        with autocast(precision, device):
            logits = model(**{k: v.to(device) for k, v in batch.items()}).logits.float()
        preds.append(logits.squeeze(-1).cpu().numpy() if reg else logits.argmax(-1).cpu().numpy())
    return glue.metrics(task, np.concatenate(preds), np.asarray(data["labels"]))


def fit(model, data: dict, cfg: RunConfig, device, pad_id: int, log=print) -> tuple[list, dict]:
    """Train the trainable parameters of `model` (the MetaTT cores). Returns (per-epoch history, training info)."""
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=cfg.lr, betas=(0.9, 0.999), eps=1e-8, weight_decay=cfg.weight_decay)
    n = len(data["train"]["labels"])
    bs = cfg.batch_size
    steps_per_epoch = math.ceil(n / bs)
    total = steps_per_epoch * cfg.epochs
    warmup = math.ceil(total * cfg.warmup_ratio)
    sched = torch.optim.lr_scheduler.LambdaLR(opt, linear_schedule(warmup, total))
    gen = torch.Generator().manual_seed(cfg.seed)
    reg = glue.num_labels(cfg.task) == 1
    frozen0 = frozen_checksum(model)

    ev = evaluate(model, data["eval"], cfg.task, device, pad_id, cfg.eval_batch_size, cfg.precision)
    history = [{"epoch": 0, "train_loss": None, "lr": 0.0, "sec": 0.0, **ev}]   # before training (frozen random head)
    log(f"epoch 0 (no training): {ev}")
    t_train = 0.0
    for epoch in range(1, cfg.epochs + 1):
        model.train()
        t0 = time.time()
        loss_sum = torch.zeros((), device=device)
        perm = torch.randperm(n, generator=gen).tolist()
        for s in range(steps_per_epoch):
            batch = glue.collate(data["train"], perm[s * bs:(s + 1) * bs], pad_id, reg)
            batch = {k: v.to(device) for k, v in batch.items()}
            with autocast(cfg.precision, device):
                loss = model(**batch).loss
            loss.backward()
            if cfg.max_grad_norm:
                torch.nn.utils.clip_grad_norm_(params, cfg.max_grad_norm)
            opt.step()
            sched.step()
            opt.zero_grad(set_to_none=True)
            loss_sum += loss.detach().float()
        if device.type == "cuda":
            torch.cuda.synchronize()
        sec = time.time() - t0
        t_train += sec
        ev = evaluate(model, data["eval"], cfg.task, device, pad_id, cfg.eval_batch_size, cfg.precision)
        row = {"epoch": epoch, "train_loss": float(loss_sum) / steps_per_epoch, "lr": sched.get_last_lr()[0],
               "sec": sec, **ev}
        history.append(row)
        log(json.dumps(row))
    info = {"steps_per_epoch": steps_per_epoch, "total_steps": total, "warmup_steps": warmup,
            "train_sec": t_train, "frozen_unchanged": frozen_checksum(model) == frozen0}
    return history, info


def summarize(history: list, task: str) -> dict:
    key = glue.main_metric(task)
    trained = [h for h in history if h["epoch"] > 0]
    best = max(trained, key=lambda h: h[key])
    out = {"metric": key, "best": best[key], "best_epoch": best["epoch"], "final": trained[-1][key],
           "before_training": history[0][key]}
    if "f1" in best:
        out["f1_at_best"] = best["f1"]
    return out


def environment(device) -> dict:
    import transformers
    env = {"python": platform.python_version(), "torch": torch.__version__, "transformers": transformers.__version__,
           "device": str(device)}
    try:
        import datasets
        env["datasets"] = datasets.__version__
    except ImportError:
        pass
    if device.type == "cuda":
        env["gpu"] = torch.cuda.get_device_name(0)
    return env


def run(cfg: RunConfig, out_dir: str) -> dict:
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    os.makedirs(out_dir, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    set_precision(cfg.precision)
    tok = AutoTokenizer.from_pretrained(cfg.model)
    data = glue.load(cfg.task, tok, cfg.max_len, cfg.limit_train, cfg.limit_eval)
    set_seed(cfg.seed)                       # the (frozen, randomly initialised) classifier head depends on the seed
    model = AutoModelForSequenceClassification.from_pretrained(cfg.model, num_labels=glue.num_labels(cfg.task))
    model = model.float()
    adapter = apply_metatt(model, MetaTT4DConfig(rank=cfg.rank, alpha=cfg.alpha,
                                                 target_modules=tuple(cfg.target_modules), init=cfg.init))
    model.to(device)
    d_in, d_out = adapter.G1.shape[0], adapter.G4.shape[1]
    params = {"trainable": count_trainable(model),
              "formula": metatt4d_num_params(d_in, d_out, adapter.G2.shape[0], adapter.G3.shape[0], cfg.rank),
              "L": adapter.G2.shape[0], "M": adapter.G3.shape[0], "D": d_in}
    assert params["trainable"] == params["formula"], params

    log_path = os.path.join(out_dir, "log.jsonl")
    with open(log_path, "w", encoding="utf-8") as logf:
        def log(msg):
            print(msg, flush=True)
            if msg.startswith("{"):
                logf.write(msg + "\n")
                logf.flush()
        log(f"run {cfg.task} seed={cfg.seed} warmup={cfg.warmup_ratio} trainable={params['trainable']} "
            f"train={len(data['train']['labels'])} eval={len(data['eval']['labels'])} device={device}")
        t0 = time.time()
        history, info = fit(model, data, cfg, device, tok.pad_token_id, log)
    result = {"cfg": asdict(cfg), "summary": summarize(history, cfg.task), "params": params, "train": info,
              "history": history, "env": environment(device), "wall_sec": time.time() - t0,
              "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
    tmp = os.path.join(out_dir, "result.json.tmp")
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(result, f, indent=1)
    os.replace(tmp, os.path.join(out_dir, "result.json"))
    return result


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--cfg", required=True, help="RunConfig as JSON")
    ap.add_argument("--out", required=True)
    a = ap.parse_args(argv)
    cfg = RunConfig.from_dict(json.loads(a.cfg))
    try:
        r = run(cfg, a.out)
    except Exception:
        os.makedirs(a.out, exist_ok=True)
        with open(os.path.join(a.out, "error.txt"), "w", encoding="utf-8") as f:
            f.write(traceback.format_exc())
        traceback.print_exc()
        return 1
    s = r["summary"]
    print(f"done: best {s['metric']} {s['best']:.2f} (epoch {s['best_epoch']}), final {s['final']:.2f}, "
          f"{r['wall_sec'] / 60:.1f} min")
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/metatt_code/metatt_repro/pipeline.py
"""Runs the minimal experiment (Table 2, RoBERTa-base, MetaTT-4D r=8, MRPC and RTE, 3 seeds) as parallel worker
processes on one GPU. Finished runs (result.json with the same config) are skipped, so the command can be repeated
after a disconnect.

  python -m metatt_repro.pipeline --out OUT [--tasks mrpc,rte] [--warmups 0.06,0.6] [--parallel 6]
  smoke test: --epochs 1 --limit_train 64 --limit_eval 64 --seeds 42
Layout: OUT/runs/<task>/w<warmup>_s<seed>/{result.json, log.jsonl, error.txt}, OUT/logs/<run>.log
"""
from __future__ import annotations

import argparse
import json
import os
import subprocess
import sys
import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from dataclasses import asdict

from . import paper
from .train import RunConfig


def build_runs(tasks, seeds, warmups, **overrides) -> list[RunConfig]:
    base = {k: v for k, v in paper.COMMON.items()}
    runs = []
    for w in warmups:
        for task in tasks:
            hp = paper.TASK_HPARAMS[task]
            for seed in seeds:
                d = dict(base, task=task, seed=seed, warmup_ratio=w, alpha=hp["alpha"], lr=hp["lr"],
                         batch_size=hp["batch_size"])
                d.update({k: v for k, v in overrides.items() if v is not None})
                runs.append(RunConfig.from_dict(d))
    return runs


def run_name(cfg: RunConfig) -> str:
    name = f"w{cfg.warmup_ratio:g}_s{cfg.seed}"
    if cfg.limit_train or cfg.limit_eval or cfg.epochs != paper.COMMON["epochs"]:
        name += f"_smoke-e{cfg.epochs}-n{cfg.limit_train}"
    if cfg.precision != "fp32":
        name += f"_{cfg.precision}"
    return name


def run_dir(out: str, cfg: RunConfig) -> str:
    return os.path.join(out, "runs", cfg.task, run_name(cfg))


def is_done(out: str, cfg: RunConfig) -> bool:
    p = os.path.join(run_dir(out, cfg), "result.json")
    if not os.path.exists(p):
        return False
    with open(p, encoding="utf-8") as f:
        old = json.load(f)["cfg"]
    return RunConfig.from_dict(old) == cfg


def prefetch(model: str, tasks) -> None:
    """Download the model and the data once, before the workers start."""
    from datasets import load_dataset
    from transformers import AutoModelForSequenceClassification, AutoTokenizer
    AutoTokenizer.from_pretrained(model)
    AutoModelForSequenceClassification.from_pretrained(model, num_labels=2)
    for t in tasks:
        load_dataset("nyu-mll/glue", t)


def _pythonpath() -> str:
    import metatt
    here = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))          # folder holding metatt_repro/
    method = os.path.dirname(os.path.dirname(os.path.abspath(metatt.__file__)))  # folder holding metatt/
    parts = [here, method] + ([os.environ["PYTHONPATH"]] if os.environ.get("PYTHONPATH") else [])
    return os.pathsep.join(dict.fromkeys(parts))


def launch(cfg: RunConfig, out: str) -> tuple[RunConfig, int, float]:
    d = run_dir(out, cfg)
    os.makedirs(d, exist_ok=True)
    os.makedirs(os.path.join(out, "logs"), exist_ok=True)
    err = os.path.join(d, "error.txt")
    if os.path.exists(err):
        os.remove(err)
    log = os.path.join(out, "logs", f"{cfg.task}_{run_name(cfg)}.log")
    env = dict(os.environ, PYTHONPATH=_pythonpath(), PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1",
               TOKENIZERS_PARALLELISM="false")
    t0 = time.time()
    with open(log, "w", encoding="utf-8") as f:
        rc = subprocess.run([sys.executable, "-m", "metatt_repro.train", "--cfg", json.dumps(asdict(cfg)),
                             "--out", d], stdout=f, stderr=subprocess.STDOUT, env=env).returncode
    return cfg, rc, time.time() - t0


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--tasks", default="mrpc,rte")
    ap.add_argument("--seeds", default=None, help="default: the paper's seeds for the model")
    ap.add_argument("--warmups", default=",".join(f"{w:g}" for w in paper.WARMUPS))
    ap.add_argument("--precision", default="fp32", choices=("fp32", "tf32", "bf16"))
    ap.add_argument("--parallel", type=int, default=1)
    ap.add_argument("--hours", type=float, default=None, help="do not start new runs after this many hours")
    ap.add_argument("--epochs", type=int, default=None)
    ap.add_argument("--limit_train", type=int, default=None)
    ap.add_argument("--limit_eval", type=int, default=None)
    ap.add_argument("--no_prefetch", action="store_true")
    a = ap.parse_args(argv)

    tasks = [t for t in a.tasks.split(",") if t]
    seeds = [int(s) for s in a.seeds.split(",")] if a.seeds else list(paper.SEEDS[paper.COMMON["model"]])
    warmups = [float(w) for w in a.warmups.split(",")]
    runs = build_runs(tasks, seeds, warmups, precision=a.precision, epochs=a.epochs,
                      limit_train=a.limit_train, limit_eval=a.limit_eval)
    todo = [r for r in runs if not is_done(a.out, r)]
    print(f"{len(runs)} runs, {len(runs) - len(todo)} already finished, {len(todo)} to run "
          f"({a.parallel} at a time) -> {a.out}", flush=True)
    if not todo:
        return 0
    if not a.no_prefetch:
        prefetch(paper.COMMON["model"], tasks)
        os.environ["HF_HUB_OFFLINE"] = "1"         # workers read the cache only (no parallel hub requests)
        os.environ["HF_DATASETS_OFFLINE"] = "1"
    deadline = time.time() + 3600 * a.hours if a.hours is not None else None
    failed = skipped = 0
    with ThreadPoolExecutor(max_workers=max(1, a.parallel)) as pool:
        futures = []
        for cfg in todo:
            futures.append(pool.submit(lambda c=cfg: launch(c, a.out) if not deadline or time.time() < deadline
                                       else (c, None, 0.0)))
        for fut in as_completed(futures):
            cfg, rc, sec = fut.result()
            tag = f"{cfg.task} warmup={cfg.warmup_ratio:g} seed={cfg.seed}"
            if rc is None:
                print(f"skipped (time budget): {tag}", flush=True)
                skipped += 1
                continue
            if rc == 0:
                with open(os.path.join(run_dir(a.out, cfg), "result.json"), encoding="utf-8") as f:
                    s = json.load(f)["summary"]
                print(f"done {tag}: best {s['metric']} {s['best']:.2f} (epoch {s['best_epoch']}), "
                      f"final {s['final']:.2f}  [{sec / 60:.1f} min]", flush=True)
            else:
                failed += 1
                print(f"FAILED {tag} (exit {rc}); last lines of the log:", flush=True)
                log = os.path.join(a.out, "logs", f"{cfg.task}_{run_name(cfg)}.log")
                with open(log, encoding="utf-8", errors="replace") as f:
                    print("".join(f.readlines()[-15:]), flush=True)
    print(f"finished: {len(todo) - failed - skipped} ok, {failed} failed, {skipped} not started (time budget; "
          f"run again to continue)", flush=True)
    return 1 if failed else 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile /content/metatt_code/metatt_repro/report.py
"""Collects OUT/runs/**/result.json and compares them with Table 2.

  python -m metatt_repro.report --out OUT
writes OUT/report/summary.csv (one row per run), OUT/report/report.md (mean +- standard error over seeds against
the paper) and OUT/report/curves.png (validation accuracy per epoch).
"""
from __future__ import annotations

import argparse
import csv
import glob
import json
import math
import os
from collections import defaultdict

from . import paper


def load_results(out: str) -> list[dict]:
    res = []
    for p in sorted(glob.glob(os.path.join(out, "runs", "*", "*", "result.json"))):
        with open(p, encoding="utf-8") as f:
            r = json.load(f)
        r["_dir"] = os.path.basename(os.path.dirname(p))
        res.append(r)
    return res


def mean_se(xs: list[float]) -> tuple[float, float]:
    n = len(xs)
    m = sum(xs) / n
    if n < 2:
        return m, float("nan")
    sd = math.sqrt(sum((x - m) ** 2 for x in xs) / (n - 1))
    return m, sd / math.sqrt(n)


def is_main(r: dict) -> bool:
    c = r["cfg"]
    return not (c.get("limit_train") or c.get("limit_eval")) and c["epochs"] == paper.COMMON["epochs"]


def rows(results: list[dict]) -> list[dict]:
    out = []
    for r in results:
        c, s = r["cfg"], r["summary"]
        out.append({"task": c["task"], "warmup": c["warmup_ratio"], "seed": c["seed"], "precision": c["precision"],
                    "smoke": not is_main(r), "best_acc": round(s["best"], 2), "best_epoch": s["best_epoch"],
                    "final_acc": round(s["final"], 2), "before_training": round(s["before_training"], 2),
                    "f1_at_best": round(s.get("f1_at_best", float("nan")), 2), "trainable": r["params"]["trainable"],
                    "head_frozen": r["train"]["frozen_unchanged"], "train_min": round(r["train"]["train_sec"] / 60, 1),
                    "gpu": r["env"].get("gpu", r["env"].get("device", "")), "torch": r["env"]["torch"],
                    "transformers": r["env"]["transformers"], "finished": r.get("finished", "")})
    return sorted(out, key=lambda x: (x["smoke"], x["warmup"], x["task"], x["seed"]))


def table(results: list[dict]) -> str:
    groups = defaultdict(list)
    for r in results:
        if is_main(r):
            c = r["cfg"]
            groups[(c["task"], c["warmup_ratio"], c["precision"])].append(r)
    lines = ["| タスク | warmup | 精度 | シード数 | 最良エポックの正解率（平均 ± 標準誤差） | 最終エポック | 論文 | 差（最良 − 論文） | 各シード（最良） |",
             "|---|---|---|---|---|---|---|---|---|"]
    for (task, w, prec), rs in sorted(groups.items(), key=lambda kv: (kv[0][1], kv[0][0])):
        best = [r["summary"]["best"] for r in rs]
        final = [r["summary"]["final"] for r in rs]
        mb, sb = mean_se(best)
        mf, sf = mean_se(final)
        p = paper.PAPER_RESULTS.get(task)
        ptxt = f"{p['value']} ± {p['se']}" if p else "–"
        diff = f"{mb - p['value']:+.2f}" if p else "–"
        seeds = ", ".join(f"{r['cfg']['seed']}: {r['summary']['best']:.2f}（{r['summary']['best_epoch']}）" for r in
                          sorted(rs, key=lambda r: r["cfg"]["seed"]))
        lines.append(f"| {task.upper()} | {w:g} | {prec} | {len(rs)} | {mb:.2f} ± {sb:.2f} | {mf:.2f} ± {sf:.2f} | "
                     f"{ptxt} | {diff} | {seeds} |")
    return "\n".join(lines)


def plot(results: list[dict], path: str) -> bool:
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
    except ImportError:
        return False
    main = [r for r in results if is_main(r)]
    if not main:
        return False
    tasks = sorted({r["cfg"]["task"] for r in main})
    fig, axes = plt.subplots(1, len(tasks), figsize=(5.5 * len(tasks), 3.8), squeeze=False)
    for ax, task in zip(axes[0], tasks):
        for r in sorted((r for r in main if r["cfg"]["task"] == task), key=lambda r: (r["cfg"]["warmup_ratio"],
                                                                                         r["cfg"]["seed"])):
            h = [x for x in r["history"] if x["epoch"] > 0]
            ls = "-" if r["cfg"]["warmup_ratio"] < 0.3 else "--"
            ax.plot([x["epoch"] for x in h], [x["accuracy"] for x in h], ls,
                    label=f"warmup {r['cfg']['warmup_ratio']:g}, seed {r['cfg']['seed']}")
        p = paper.PAPER_RESULTS.get(task)
        if p:
            ax.axhline(p["value"], color="k", lw=0.8)
            ax.axhspan(p["value"] - p["se"], p["value"] + p["se"], color="k", alpha=0.1)
        ax.set_title(f"{task.upper()} (RoBERTa-base, MetaTT-4D r=8)")
        ax.set_xlabel("epoch")
        ax.set_ylabel("validation accuracy (%)")
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(path, dpi=130)
    plt.close(fig)
    return True


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    a = ap.parse_args(argv)
    results = load_results(a.out)
    rep = os.path.join(a.out, "report")
    os.makedirs(rep, exist_ok=True)
    if not results:
        print("no finished runs under", a.out)
        return 1
    rs = rows(results)
    with open(os.path.join(rep, "summary.csv"), "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(rs[0]))
        w.writeheader()
        w.writerows(rs)
    md = ["# MetaTT-4D（r=8）× RoBERTa-base：論文の表 2 との比較", "",
          "論文の値は表 2（v2・v3 で同じ）。括弧ではなく ± で標準誤差を示す。正解率は GLUE の検証データ。", "",
          table(results), "",
          f"- 終わったラン：{sum(not r['smoke'] for r in rs)} 本（スモーク {sum(r['smoke'] for r in rs)} 本）",
          f"- 学習したパラメータ数：{sorted({r['trainable'] for r in rs})}（論文 {paper.PAPER_PARAMS_K}k）",
          f"- 分類ヘッドが学習で変わっていない：{all(r['head_frozen'] for r in rs)}",
          f"- 参考（論文の値ではない）：多数派クラスだけを答えたときの正解率 "
          + "、".join(f"{t.upper()} {v}" for t, v in paper.MAJORITY_BASELINE.items())
          + "。最良の値がこれと同じランは、何も学習できていない",
          f"- 環境：{sorted({r['gpu'] for r in rs})}、torch {sorted({r['torch'] for r in rs})}、"
          f"transformers {sorted({r['transformers'] for r in rs})}"]
    with open(os.path.join(rep, "report.md"), "w", encoding="utf-8") as f:
        f.write("\n".join(md) + "\n")
    plotted = plot(results, os.path.join(rep, "curves.png"))
    print("\n".join(md))
    print("\nwrote", os.path.join(rep, "summary.csv"), os.path.join(rep, "report.md"),
          os.path.join(rep, "curves.png") if plotted else "(no plot: matplotlib missing or no full runs)")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/metatt_code/metatt_repro/count.py
"""Checks the trainable-parameter counts of the MetaTT-4D rows of Table 2 without training or downloading weights:
builds RoBERTa from its config (on the meta device, so no memory is used), applies MetaTT-4D and counts.

  python -m metatt_repro.count [--out param_check.md]
"""
from __future__ import annotations

import argparse
import contextlib

import torch

from . import paper
from metatt import MetaTT4DConfig, apply_metatt, count_trainable, metatt4d_num_params


def build(model_name: str):
    from transformers import AutoConfig, AutoModelForSequenceClassification
    cfg = AutoConfig.from_pretrained(model_name, num_labels=2)
    for ctx in (torch.device("meta"), contextlib.nullcontext()):   # fall back to CPU if meta is not supported
        try:
            with ctx:
                return AutoModelForSequenceClassification.from_config(cfg), cfg
        except Exception:
            continue
    raise RuntimeError(f"cannot build {model_name}")


def check() -> list[dict]:
    out = []
    for (name, rank), printed_k in paper.TABLE2_PARAMS_K.items():
        model, hf_cfg = build(name)                    # fresh model per row (apply_metatt modifies it)
        apply_metatt(model, MetaTT4DConfig(rank=rank, target_modules=paper.COMMON["target_modules"]))
        n = count_trainable(model)
        d, L = hf_cfg.hidden_size, hf_cfg.num_hidden_layers
        f = metatt4d_num_params(d, d, L, 2, rank)
        out.append({"model": name.split("/")[-1], "rank": rank, "counted": n, "formula": f, "paper_k": printed_k,
                    "match": n == f and round(n / 1000) == printed_k})
    return out


def main(argv=None) -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", default=None)
    a = ap.parse_args(argv)
    rows = check()
    lines = ["| モデル | ランク | 数えた値 | 式 2Dr+(L+M)r² | 論文（×10³） | 一致 |", "|---|---|---|---|---|---|"]
    lines += [f"| {r['model']} | {r['rank']} | {r['counted']:,} | {r['formula']:,} | {r['paper_k']} | "
              f"{'✓' if r['match'] else '✗'} |" for r in rows]
    text = "# 表 2 の MetaTT-4D のパラメータ数（Q・V に適用、M = 2）\n\n" + "\n".join(lines) + "\n"
    print(text)
    if a.out:
        import os
        os.makedirs(os.path.dirname(os.path.abspath(a.out)), exist_ok=True)
        with open(a.out, "w", encoding="utf-8") as f:
            f.write(text)
    return 0 if all(r["match"] for r in rows) else 1


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile /content/metatt_code/tests/test_core.py
"""CPU tests for the MetaTT-4D implementation (no download, no GPU).

  python reproduction/tests/test_core.py        (or: python -m pytest reproduction/tests)
"""
import math
import os
import sys

import torch
from torch import nn

HERE = os.path.dirname(os.path.abspath(__file__))
sys.path.insert(0, os.path.dirname(HERE))                                        # reproduction/ (metatt_repro)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.dirname(HERE)), "method"))  # method/ (metatt)

from metatt import MetaTT4D, MetaTT4DConfig, MetaTTLinear, apply_metatt, count_trainable, metatt4d_num_params  # noqa: E402
from metatt_repro import glue  # noqa: E402
from metatt_repro.train import RunConfig, fit, linear_schedule  # noqa: E402


def roberta(hidden=32, layers=3, heads=4, num_labels=2, vocab=100):
    from transformers import RobertaConfig, RobertaForSequenceClassification
    cfg = RobertaConfig(vocab_size=vocab, hidden_size=hidden, num_hidden_layers=layers, num_attention_heads=heads,
                        intermediate_size=2 * hidden, max_position_embeddings=64, num_labels=num_labels)
    torch.manual_seed(0)
    return RobertaForSequenceClassification(cfg)


def batch(n=4, t=10, vocab=100):
    g = torch.Generator().manual_seed(1)
    ids = torch.randint(3, vocab, (n, t), generator=g)
    return {"input_ids": ids, "attention_mask": torch.ones_like(ids)}


def test_param_count_roberta_base():
    """Table 2: RoBERTa-base (D=768, L=12), Q and V (M=2): r=8 -> 13,184 (13k), r=24 -> 44,928 (45k), r=64 -> 155,648 (156k)."""
    from transformers import RobertaConfig, RobertaForSequenceClassification
    cfg = RobertaConfig(num_labels=2)                     # defaults are the roberta-base sizes
    for r, k in ((8, 13), (24, 45), (64, 156)):
        with torch.device("meta"):
            m = RobertaForSequenceClassification(cfg)
        apply_metatt(m, MetaTT4DConfig(rank=r))
        n = count_trainable(m)
        assert n == metatt4d_num_params(768, 768, 12, 2, r) == 2 * 768 * r + (12 + 2) * r * r
        assert round(n / 1000) == k, (r, n)
    assert metatt4d_num_params(768, 768, 12, 2, 4) == 6368      # the hyper-parameter table's "rank 4" cannot be 13k


def test_init_ze_id_id_id_and_zero_update():
    m = roberta()
    m.eval()
    x = batch()
    before = m(**x).logits
    ad = apply_metatt(m, MetaTT4DConfig(rank=8, alpha=4.0))
    assert torch.count_nonzero(ad.G1) == 0
    for s in list(ad.G2) + list(ad.G3):
        assert torch.equal(s, torch.eye(8))
    assert torch.equal(ad.G4, torch.eye(8, 32))           # nn.init.eye_ on the r x D output core
    assert torch.allclose(m(**x).logits, before)          # Delta W = 0 at initialisation


def test_only_cores_train_and_head_stays_frozen():
    m = roberta()
    ad = apply_metatt(m, MetaTT4DConfig(rank=4))
    names = [n for n, p in m.named_parameters() if p.requires_grad]
    assert names == ["metatt.G1", "metatt.G2", "metatt.G3", "metatt.G4"], names
    assert all(not p.requires_grad for n, p in m.named_parameters() if n.startswith("classifier"))
    assert not any("metatt" in k for k in m.state_dict() if not k.startswith("metatt."))  # cores stored once


def test_layer_and_type_slices():
    m = roberta(layers=3)
    apply_metatt(m, MetaTT4DConfig(rank=4, target_modules=("query", "value")))
    seen = {}
    for name, mod in m.named_modules():
        if isinstance(mod, MetaTTLinear):
            seen[name] = (mod.layer, mod.mtype)
    assert len(seen) == 6
    assert seen["roberta.encoder.layer.2.attention.self.value"] == (2, 1)
    assert seen["roberta.encoder.layer.0.attention.self.query"] == (0, 0)
    assert isinstance(m.roberta.encoder.layer[1].attention.self.key, nn.Linear)   # K is not adapted


def test_forward_equals_dense_delta_and_paper_pseudocode():
    """Our adapter term equals x @ Delta W and the paper's Algorithm 3/4 (ParameterList cores, typos fixed)."""
    torch.manual_seed(2)
    cfg = MetaTT4DConfig(rank=(3, 4, 5), alpha=0.5, init="no-no-no-no")
    ad = MetaTT4D(16, 16, 3, 2, cfg)
    base = nn.Linear(16, 16)
    lin = MetaTTLinear(base, ad, layer=2, mtype=1)
    x = torch.randn(7, 16)
    ref_dense = base(x) + x @ ad.delta_weight(2, 1)
    M1, M2, M3, M4 = ad.G1, nn.ParameterList([nn.Parameter(s) for s in ad.G2])[2], \
        nn.ParameterList([nn.Parameter(s) for s in ad.G3])[1], ad.G4
    ref_paper = base(x) + cfg.alpha * (((x @ M1) @ M2) @ M3) @ M4      # Algorithm 4
    assert torch.allclose(lin(x), ref_dense, atol=1e-5)
    assert torch.allclose(lin(x), ref_paper, atol=1e-5)


def test_first_step_moves_only_g1():
    """With G1 = 0 the first gradient reaches only G1 (the other cores' gradients contain x G1 = 0)."""
    m = roberta()
    ad = apply_metatt(m, MetaTT4DConfig(rank=4, alpha=4.0))
    x = batch()
    out = m(**x, labels=torch.tensor([0, 1, 0, 1]))
    out.loss.backward()
    assert ad.G1.grad.abs().sum() > 0
    for g in (ad.G2.grad, ad.G3.grad, ad.G4.grad):
        assert g.abs().sum() == 0


def test_linear_schedule_matches_trainer():
    f = linear_schedule(10, 100)
    assert f(0) == 0 and f(5) == 0.5 and f(10) == 1.0 and math.isclose(f(55), 0.5) and f(100) == 0.0


def test_metrics():
    import numpy as np
    p, y = np.array([1, 1, 0, 0, 1]), np.array([1, 0, 0, 1, 1])
    m = glue.metrics("mrpc", p, y)
    assert math.isclose(m["accuracy"], 60.0) and math.isclose(m["f1"], 100 * 2 * 2 / (2 * 2 + 1 + 1))
    assert math.isclose(glue.metrics("stsb", np.array([1., 2., 3.]), np.array([3., 5., 9.]))["spearman"], 100.0)


def test_fit_smoke():
    """Two epochs of the real training loop on random tokens: runs, changes the cores, leaves the rest frozen."""
    m = roberta()
    ad = apply_metatt(m, MetaTT4DConfig(rank=4, alpha=4.0))
    g = torch.Generator().manual_seed(3)
    ids = [torch.randint(3, 100, (int(torch.randint(4, 12, (1,), generator=g)),), generator=g).tolist() for _ in range(40)]
    labels = [int(sum(s) % 2) for s in ids]
    data = {"train": {"input_ids": ids[:32], "labels": labels[:32]}, "eval": {"input_ids": ids[32:], "labels": labels[32:]}}
    cfg = RunConfig(task="rte", seed=0, alpha=4.0, lr=1e-2, batch_size=8, warmup_ratio=0.06, rank=4, epochs=2)
    g1 = ad.G1.detach().clone()
    hist, info = fit(m, data, cfg, torch.device("cpu"), pad_id=1, log=lambda s: None)
    assert [h["epoch"] for h in hist] == [0, 1, 2]
    assert info["frozen_unchanged"] and info["total_steps"] == 8 and info["warmup_steps"] == 1
    assert not torch.equal(ad.G1.detach(), g1)


if __name__ == "__main__":
    tests = [v for k, v in sorted(globals().items()) if k.startswith("test_")]
    for t in tests:
        t()
        print("ok", t.__name__)
    print(f"all {len(tests)} tests passed")


## 準備（実装のテスト）

In [ ]:
import json, glob
os.chdir(CODE)
sys.path.insert(0, CODE)

def sh(cmd):
    """コマンドを実行し、出力をこのセルに流す（ダウンロードの進捗表示などは省く）。"""
    noise = ('it/s]', 's/it]', 'Loading weights', 'LOAD REPORT', 'UNEXPECTED', 'MISSING', 'Notes:', 'can be ignored',
             'newly initialized', '---+---', 'Key ', 'HF_TOKEN', 'Redirects are currently')
    env = dict(os.environ, PYTHONIOENCODING='utf-8', PYTHONUNBUFFERED='1')
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
                         encoding='utf-8', errors='replace', env=env)
    for line in p.stdout:
        if not any(s in line for s in noise):
            print(line, end='', flush=True)
    return p.wait()

# 実装のテスト（CPU、1 分ほど）：パラメータ数、初期状態で ΔW = 0、分類ヘッドが凍結、論文の擬似コードと同じ出力、など
r = sh(f'{sys.executable} tests/test_core.py')
assert r == 0, 'テストが通りませんでした（上のエラーを見てください）'

## 実行

In [ ]:
# ステップ 0：表 2 の MetaTT-4D のパラメータ数の照合（学習なし）
sh(f'{sys.executable} -m metatt_repro.count --out "{OUT}/report/param_check.md"')
# ステップ 1：12 本のラン（終わったものは飛ばす）
t0 = time.time()
sh(f'{sys.executable} -m metatt_repro.pipeline --out "{OUT}" --tasks {TASKS} --warmups {WARMUPS} '
   f'--precision {PRECISION} --parallel {PARALLEL} --hours {HOURS}')
print(f'経過 {(time.time() - t0) / 60:.1f} 分')

## 結果（論文の表 2 との比較）

In [ ]:
from IPython.display import display, Markdown, Image
r = sh(f'{sys.executable} -m metatt_repro.report --out "{OUT}" > /dev/null')
rep = f'{OUT}/report/report.md'
if os.path.exists(rep):
    display(Markdown(open(rep, encoding='utf-8').read()))
if os.path.exists(f'{OUT}/report/curves.png'):
    display(Image(f'{OUT}/report/curves.png'))
print('ランごとの値:', f'{OUT}/report/summary.csv', '| パラメータ数の照合:', f'{OUT}/report/param_check.md')

## うまくいかないとき
ランが失敗する・終わらないときは、このセルを実行して、出てきた表示をそのまま共有してください。
各ランのログの最後と、失敗したランのエラー（`error.txt`）を表示します。

In [ ]:
import glob, os
for f in sorted(glob.glob(f'{OUT}/logs/*.log')):
    print('=====', f)
    print(open(f, encoding='utf-8', errors='replace').read()[-2000:])
errs = glob.glob(f'{OUT}/runs/**/error.txt', recursive=True)
print(f'\n失敗したランの記録: {len(errs)} 件')
for f in errs[:3]:
    print('===== ERROR FILE', f)
    print(open(f, encoding='utf-8', errors='replace').read()[-2500:])